In [ ]:
import pandas as pd

# Load the original dataset
df = pd.read_csv("mtsamples.csv")

# Check dataset dimensions
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

# Display columns
print("\nColumn names:")
print(df.columns.tolist())

# Display the first five rows
display(df.head())


In [ ]:
# Count missing values in every column
print("Missing values:")
print(df.isnull().sum())

# Calculate percentages
print("\nMissing percentages:")
print((df.isnull().mean() * 100).round(2))


In [ ]:
# Count rows with no transcription
missing_count = df["transcription"].isna().sum()

# Keep rows that have a transcription
valid_df = df.dropna(subset=["transcription"]).copy()

# Count exact duplicates of transcription text
duplicate_count = valid_df.duplicated(
    subset=["transcription"]
).sum()

unique_count = valid_df["transcription"].nunique()

print("Original rows:", len(df))
print("Missing transcriptions:", missing_count)
print("Rows with transcription:", len(valid_df))
print("Duplicate transcriptions:", duplicate_count)
print("Unique transcriptions:", unique_count)


In [ ]:
# Find duplicate transcription groups
duplicates = valid_df[
    valid_df.duplicated(
        subset=["transcription"],
        keep=False
    )
].copy()

# Show the first 10 matching records
display(
    duplicates[
        ["sample_name", "medical_specialty", "transcription"]
    ].head(10)
)

print("Rows belonging to duplicate groups:", len(duplicates))


In [ ]:
# Count words in each transcription
valid_df["word_count"] = (
    valid_df["transcription"]
    .astype(str)
    .str.split()
    .str.len()
)

print("Transcription length statistics:")
print(valid_df["word_count"].describe())

print("\nTranscriptions shorter than 20 words:")
print((valid_df["word_count"] < 20).sum())

print("\nTranscriptions longer than 2000 words:")
print((valid_df["word_count"] > 2000).sum())


In [ ]:
# Start from the original dataset
df_clean = df.copy()

# Remove the unnecessary index column
df_clean = df_clean.drop(columns=["Unnamed: 0"], errors="ignore")

# Remove rows with missing transcription
df_clean = df_clean.dropna(subset=["transcription"])

# Remove rows with transcription containing only spaces
df_clean = df_clean[
    df_clean["transcription"].astype(str).str.strip() != ""
].copy()

# Reset row indices
df_clean = df_clean.reset_index(drop=True)

print("Rows after basic cleaning:", len(df_clean))
print("Columns:", df_clean.columns.tolist())


In [ ]:
import re

# Remove leading and trailing spaces from metadata
for col in ["description", "medical_specialty", "sample_name", "keywords"]:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Normalize whitespace in transcription
def clean_transcription(text):
    text = str(text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

df_clean["transcription"] = df_clean[
    "transcription"
].apply(clean_transcription)

print("Whitespace normalization completed.")
print("Remaining rows:", len(df_clean))


In [ ]:
# Save all records belonging to duplicate groups
duplicate_records = df_clean[
    df_clean.duplicated(
        subset=["transcription"],
        keep=False
    )
].copy()

duplicate_records.to_csv(
    "duplicate_records.csv",
    index=False
)

# Keep one record for each exact transcription
df_unique = df_clean.drop_duplicates(
    subset=["transcription"],
    keep="first"
).copy()

df_unique = df_unique.reset_index(drop=True)

print("Rows before deduplication:", len(df_clean))
print("Rows after deduplication:", len(df_unique))
print("Duplicates removed:", len(df_clean) - len(df_unique))
print("Duplicate review file saved.")


In [ ]:
# Calculate word count after cleaning
df_unique["word_count"] = (
    df_unique["transcription"]
    .str.split()
    .str.len()
)

# Select short transcriptions
short_records = df_unique[
    df_unique["word_count"] < 20
].copy()

print("Short records after deduplication:", len(short_records))

# Display all short records
display(
    short_records[
        ["sample_name", "medical_specialty",
         "word_count", "transcription"]
    ]
)

# Save short records for review
short_records.to_csv(
    "short_records_review.csv",
    index=False
)


In [ ]:
# Keep the meaningful short EEG record
keep_short = ["Video EEG - 2"]

# Identify short records to exclude
excluded_short = df_unique[
    (df_unique["word_count"] < 20) &
    (~df_unique["sample_name"].isin(keep_short))
].copy()

# Save excluded records for documentation
excluded_short.to_csv(
    "excluded_short_records.csv",
    index=False
)

# Create the working dataset
df_ready = df_unique.drop(
    index=excluded_short.index
).copy()

df_ready = df_ready.reset_index(drop=True)

print("Unique records before filtering:", len(df_unique))
print("Short records excluded:", len(excluded_short))
print("Records remaining:", len(df_ready))

print("\nMinimum word count:",
      df_ready["word_count"].min())


In [ ]:
# Check specialty differences within duplicate groups
specialty_check = (
    df_clean.groupby("transcription")
    ["medical_specialty"]
    .nunique()
)

# Find transcripts assigned to multiple specialties
conflicting_texts = specialty_check[
    specialty_check > 1
].index

conflicting_records = df_clean[
    df_clean["transcription"].isin(conflicting_texts)
].copy()

print(
    "Unique transcriptions with multiple specialty labels:",
    len(conflicting_texts)
)

print(
    "Total records in conflicting groups:",
    len(conflicting_records)
)

# Display examples
display(
    conflicting_records[
        ["sample_name", "medical_specialty"]
    ].head(20)
)

# Save for review
conflicting_records.to_csv(
    "specialty_conflicts.csv",
    index=False
)


In [ ]:

# Select one transcription having multiple specialty labels
example_text = conflicting_texts[0]

# Retrieve every occurrence of that transcription
example_group = df_clean[
    df_clean["transcription"] == example_text
]

print("Number of duplicate records:", len(example_group))

print("\nSpecialty labels:")
print(example_group["medical_specialty"].unique())

print("\nSample names:")
print(example_group["sample_name"].tolist())

print("\nTranscription preview:")
print(example_text[:600])


In [ ]:
import pandas as pd

# Collect all specialties associated with each transcription
specialty_map = (
    df_clean.groupby("transcription")["medical_specialty"]
    .apply(lambda x: " | ".join(sorted(x.dropna().unique())))
    .to_dict()
)

# Add all original specialty labels
df_ready["all_specialties"] = (
    df_ready["transcription"].map(specialty_map)
)

# Give each record a unique case ID
df_ready["case_id"] = [
    f"CASE_{i:04d}" for i in range(1, len(df_ready) + 1)
]

# Arrange columns
columns = [
    "case_id",
    "medical_specialty",
    "all_specialties",
    "sample_name",
    "description",
    "transcription",
    "keywords",
    "word_count"
]

df_ready = df_ready[columns]

# Save the dataset
df_ready.to_csv(
    "mtsamples_cleaned.csv",
    index=False
)

# Verify the saved file
check_df = pd.read_csv("mtsamples_cleaned.csv")

print("Cleaned dataset saved successfully!")
print("Total records:", len(check_df))
print("Total columns:", len(check_df.columns))
print("Missing transcriptions:",
      check_df["transcription"].isna().sum())
print("Duplicate transcriptions:",
      check_df.duplicated(subset=["transcription"]).sum())

display(check_df.head())


In [ ]:
# Count records for each primary medical specialty
specialty_counts = (
    df_ready["medical_specialty"]
    .value_counts()
)

print("Total cleaned records:", len(df_ready))
print("Number of primary specialties:",
      df_ready["medical_specialty"].nunique())

print("\nRecords per specialty:")
print(specialty_counts.to_string())


In [ ]:
# Calculate each specialty's percentage
specialty_distribution = (
    df_ready["medical_specialty"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Top 10 specialties by percentage:")
print(specialty_distribution.head(10).to_string())

print("\nSpecialties with fewer than 10 records:")
small_specialties = specialty_counts[
    specialty_counts < 10
]

print(small_specialties.to_string())

print("\nNumber of specialties with fewer than 10 records:",
      len(small_specialties))


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# Use the cleaned dataset
texts = df_ready["transcription"].fillna("").astype(str)

# Convert clinical text into TF-IDF vectors
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(texts)

# Calculate similarity between records
similarity_matrix = cosine_similarity(tfidf_matrix)

# Exclude comparison of a record with itself
np.fill_diagonal(similarity_matrix, 0)

# Collect highly similar pairs
similar_pairs = []

threshold = 0.90

for i in range(len(df_ready)):
    for j in range(i + 1, len(df_ready)):
        if similarity_matrix[i, j] >= threshold:
            similar_pairs.append({
                "case_id_1": df_ready.iloc[i]["case_id"],
                "case_id_2": df_ready.iloc[j]["case_id"],
                "sample_1": df_ready.iloc[i]["sample_name"],
                "sample_2": df_ready.iloc[j]["sample_name"],
                "similarity": round(
                    float(similarity_matrix[i, j]), 4
                )
            })

similar_df = pd.DataFrame(
    similar_pairs,
    columns=[
        "case_id_1",
        "case_id_2",
        "sample_1",
        "sample_2",
        "similarity"
    ]
)

similar_df = similar_df.sort_values(
    "similarity",
    ascending=False
)

print("Total records checked:", len(df_ready))
print("Similarity threshold:", threshold)
print("Highly similar pairs found:", len(similar_df))

display(similar_df.head(20))

# Save the review file
similar_df.to_csv(
    "near_duplicate_pairs.csv",
    index=False
)

print("\nNear-duplicate review file saved.")


In [ ]:
# Union-find keeps connected near-duplicate pairs together
case_ids = df_ready["case_id"].tolist()
parent = {case_id: case_id for case_id in case_ids}

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a

# Join cases that have high similarity
for _, pair in similar_df.iterrows():
    union(pair["case_id_1"], pair["case_id_2"])

# Give each record a group identifier
df_ready["similarity_group"] = [
    find(case_id) for case_id in case_ids
]

print("Total records:", len(df_ready))
print(
    "Total similarity groups:",
    df_ready["similarity_group"].nunique()
)

print(
    "Records in groups with multiple cases:",
    df_ready["similarity_group"].map(
        df_ready["similarity_group"].value_counts()
    ).gt(1).sum()
)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# First split: 70% development, 30% temporary holdout
splitter1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42
)

train_idx, temp_idx = next(
    splitter1.split(
        df_ready,
        groups=df_ready["similarity_group"]
    )
)

train_df = df_ready.iloc[train_idx].copy()
temp_df = df_ready.iloc[temp_idx].copy()

# Second split: divide holdout into validation and test
splitter2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    splitter2.split(
        temp_df,
        groups=temp_df["similarity_group"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# Check group leakage
train_groups = set(train_df["similarity_group"])
val_groups = set(val_df["similarity_group"])
test_groups = set(test_df["similarity_group"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

# Save partitions
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))
print("Total records:",
      len(train_df) + len(val_df) + len(test_df))

print("\nSimilarity group leakage: 0")
